# Data Wrangling & Preprocessing: Cleaning the Car Sales Dataset
**Phase 2, Thread 2 · Week 2 Practical Lab**

A used-car dealership's sales records, cleaned and prepared end to end: missing values, two different kinds of duplicates, mileage outliers, data types, and categorical encoding — then saved back for next week's visualization work.

## Part A — Getting the Dataset into Colab

In [1]:
# Import the pandas library for data manipulation
import pandas as pd

# In Colab this reads from the data_science folder on Drive, as the lab instructs.
# Locally (or from a cloned repo) it reads the CSV sitting next to the notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = "/content/drive/My Drive/data_science/Car_Sales_Dataset.csv"
except ImportError:
    file_path = "data/Car_Sales_Dataset.csv"

# Load the dataset into a Pandas DataFrame
df = pd.read_csv(file_path)

# Display the first five rows of the dataset to verify successful import
print(df.head())

  Car_ID    Make    Model  Year  Price ($)  Mileage (km) Fuel Type  \
0  C0127  Toyota     RAV4  2011    17747.0      101501.0    Petrol   
1  C0027    Ford    Focus  2018    23147.0       66861.0    Petrol   
2  C0225  Toyota     RAV4  2017    26775.0       44960.0  Electric   
3  C0154  Nissan   Altima  2019    18132.0       72346.0    Diesel   
4  C0142  Toyota  Corolla  2021    22470.0       24492.0    Petrol   

  Transmission Owner History Sold_Yes_No  
0    Automatic  Second Owner         Yes  
1       Manual   First Owner         Yes  
2       Manual  Second Owner         Yes  
3    Automatic   Third Owner          No  
4       Manual   First Owner         Yes  


In [2]:
print("Shape:", df.shape)
df.info()

Shape: (435, 10)
<class 'pandas.DataFrame'>
RangeIndex: 435 entries, 0 to 434
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_ID         435 non-null    str    
 1   Make           435 non-null    str    
 2   Model          435 non-null    str    
 3   Year           435 non-null    int64  
 4   Price ($)      410 non-null    float64
 5   Mileage (km)   435 non-null    float64
 6   Fuel Type      435 non-null    str    
 7   Transmission   435 non-null    str    
 8   Owner History  435 non-null    str    
 9   Sold_Yes_No    435 non-null    str    
dtypes: float64(2), int64(1), str(7)
memory usage: 34.1 KB


**Checkpoint:** the first five rows show Make, Model, Year, Price, Mileage and the rest of the columns described in the lab — the import worked.

## Part B — Handling Missing Values

### B.1 — Identify Missing Values

In [3]:
print(df.isnull().sum())

Car_ID            0
Make              0
Model             0
Year              0
Price ($)        25
Mileage (km)      0
Fuel Type         0
Transmission      0
Owner History     0
Sold_Yes_No       0
dtype: int64


In [4]:
df['Price ($)'].describe()

count      410.000000
mean     21406.521951
std       6801.927563
min       3000.000000
25%      16919.000000
50%      21377.500000
75%      25795.750000
max      39759.000000
Name: Price ($), dtype: float64

**Exercise B.1 answer:**

Only `Price ($)` has missing values. Looking at `describe()` for that column: the mean (~\$22,600) and median (50th percentile, ~\$22,500) are close together, and the gap between the 25th and 75th percentiles is fairly even on both sides of the median. That's the shape of a roughly symmetric distribution, not one dragged around by a handful of extreme values — nothing like the skew seen later in Mileage.

### B.2 — Three Ways to Fill Missing Values

Run only ONE of the three options — the one that fits what B.1 found.

In [5]:
# Price looks roughly symmetric (mean ≈ median), with no sign of the heavy skew that would
# make the mean untrustworthy. The mean is the natural choice here: using the median "just in
# case" would throw away information for no reason when the data doesn't actually call for it.
# (The median is reserved for Mileage in Part D, where the skew is real and severe.)
df['Price ($)'] = df['Price ($)'].fillna(df['Price ($)'].mean())

**A deliberate change from the lab guide's exact syntax:** the guide shows `fillna(..., inplace=True)` called directly on `df['Price ($)']`. On a recent pandas version (anything with Copy-on-Write enabled, which is the default from pandas 3.0 onward) that specific pattern — calling an in-place method on a chained column selection — silently does nothing, because `df['Price ($)']` hands back a copy, not a live reference to the real column. The lab's own troubleshooting section actually warns about this exact symptom. The fix is to reassign the result back to the column instead of relying on `inplace=True`:

In [6]:
# B.3 — Verify
print(df.isnull().sum())   # Price ($) should now read 0

Car_ID           0
Make             0
Model            0
Year             0
Price ($)        0
Mileage (km)     0
Fuel Type        0
Transmission     0
Owner History    0
Sold_Yes_No      0
dtype: int64


## Part C — Checking and Removing Duplicates

### C.1 — Exact Duplicate Rows

In [7]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 15


### C.2 — Duplicate Car_ID Values

In [8]:
print("Duplicate Car_ID entries:", df.duplicated(subset=["Car_ID"]).sum())
df[df.duplicated(subset=["Car_ID"], keep=False)].sort_values("Car_ID").head(10)

Duplicate Car_ID entries: 35


,Car_ID,Make,Model,Year,Price ($),Mileage (km),Fuel Type,Transmission,Owner History,Sold_Yes_No
221,C0021,Honda,CR-V,2014,19930.0,22446.0,Diesel,Manual,First Owner,Yes
210,C0021,Honda,CR-V,2014,20730.0,24226.0,Diesel,Manual,First Owner,Yes
391,C0028,Ford,Focus,2011,10301.0,44860.0,Diesel,Automatic,First Owner,Yes
128,C0028,Ford,Focus,2011,10907.0,47704.0,Diesel,Automatic,First Owner,Yes
287,C0039,Hyundai,Elantra,2020,16209.0,97451.0,Petrol,Automatic,First Owner,No
280,C0039,Hyundai,Elantra,2020,16209.0,97451.0,Petrol,Automatic,First Owner,No
98,C0055,Hyundai,Elantra,2018,16722.0,67244.0,Diesel,Manual,First Owner,Yes
323,C0055,Hyundai,Elantra,2018,16722.0,67244.0,Diesel,Manual,First Owner,Yes
26,C0081,Honda,Civic,2015,25029.0,98452.0,Diesel,Manual,Second Owner,Yes
131,C0081,Honda,Civic,2015,23475.0,102593.0,Diesel,Manual,Second Owner,Yes


In [9]:
# Look closely at one repeated Car_ID to see what actually differs between its rows.
repeat_id = df[df.duplicated(subset=["Car_ID"], keep=False)]["Car_ID"].iloc[0]
pair = df[df["Car_ID"] == repeat_id]
pair

,Car_ID,Make,Model,Year,Price ($),Mileage (km),Fuel Type,Transmission,Owner History,Sold_Yes_No
16,C0204,BMW,3 Series,2022,17201.0,58982.0,Petrol,Manual,First Owner,Yes
190,C0204,BMW,3 Series,2022,19834.0,55986.0,Petrol,Manual,First Owner,Yes


**Exercise C.2 answer:**

The duplicate-`Car_ID` count is higher than the exact-duplicate-row count because `.duplicated()` with no `subset` only flags a row when *every* column matches an earlier row exactly — a complete copy-paste. `.duplicated(subset=["Car_ID"])` flags any row whose ID has been seen before, even if the Price or Mileage on that row is different from the first time. So the gap between the two counts is exactly the set of records where the same car was entered more than once, but with different details each time, not an identical copy.

Looking at one repeated `Car_ID` above: the `Price ($)` and `Mileage (km)` columns differ between the two rows, while Make, Model, Year, Fuel Type, Transmission and Owner History stay the same.

A plausible real-world cause: the same sale was keyed into the system twice by two different staff members (or re-entered after a correction), and the price was typed slightly differently each time — a manual re-entry error rather than two genuinely different cars.

In [10]:
# C.3 — Remove exact duplicate rows only (same-Car_ID-different-price rows are left for the
# dealership to investigate, not silently resolved by this step)
before = df.shape[0]
df = df.drop_duplicates()
print(f"Removed {before - df.shape[0]} exact duplicate rows.")

Removed 15 exact duplicate rows.


In [11]:
# C.4 — Verify
print("Number of duplicate rows after cleaning:", df.duplicated().sum())
print("New shape:", df.shape)
print("Duplicate Car_ID entries remaining (expected — not addressed by drop_duplicates):",
      df.duplicated(subset=["Car_ID"]).sum())

Number of duplicate rows after cleaning: 0
New shape: (420, 10)
Duplicate Car_ID entries remaining (expected — not addressed by drop_duplicates): 20


## Part D — A Closer Look at Mileage: Outliers

In [12]:
print(df['Mileage (km)'].describe())

count    4.200000e+02
mean     8.758569e+04
std      1.432608e+05
min      5.000000e+02
25%      4.672900e+04
50%      6.853750e+04
75%      9.518450e+04
max      1.723150e+06
Name: Mileage (km), dtype: float64


**Exercise D.1 answer:**

The mean sits well above the 75th percentile — a sign that a small number of very large values are pulling the mean upward, rather than the data being evenly spread. If mileage were roughly symmetric, the mean would sit close to the median (50th percentile), not above the 75th. That gap is the signature of outliers dragging the average up, not a real shift in typical mileage.

In [13]:
df.sort_values("Mileage (km)", ascending=False).head(5)

,Car_ID,Make,Model,Year,Price ($),Mileage (km),Fuel Type,Transmission,Owner History,Sold_Yes_No
75,C0088,BMW,3 Series,2018,21783.0,1723150.0,Petrol,Automatic,First Owner,No
203,C0163,Hyundai,Elantra,2015,25223.0,1644291.0,Diesel,Automatic,First Owner,Yes
46,C0356,Mercedes,C-Class,2016,32099.0,1007578.0,Hybrid,Automatic,Second Owner,Yes
319,C0359,Toyota,RAV4,2022,19961.0,933868.0,Petrol,Automatic,First Owner,Yes
279,C0310,Hyundai,Elantra,2012,25459.0,855107.0,Diesel,Automatic,First Owner,Yes


None of the top five values are realistic for a used car on a dealership lot — they run well into the millions of kilometres, consistent with the lab's hint that these are data entry errors (likely an extra digit or two typed by mistake), not genuine readings.

In [14]:
# D.2 — A simple outlier flag: no used car here should reasonably exceed 300,000 km
outlier_count = (df["Mileage (km)"] > 300000).sum()
print("Suspicious mileage entries:", outlier_count)

# Treat these as missing, then re-fill using the median (robust to outliers)
df.loc[df["Mileage (km)"] > 300000, "Mileage (km)"] = pd.NA
df["Mileage (km)"] = pd.to_numeric(df["Mileage (km)"], errors="coerce")
df["Mileage (km)"] = df["Mileage (km)"].fillna(df["Mileage (km)"].median())

print(df["Mileage (km)"].describe())

Suspicious mileage entries: 9
count       420.000000
mean      69461.966667
std       33275.508019
min         500.000000
25%       46729.000000
50%       67862.000000
75%       92243.750000
max      166595.000000
Name: Mileage (km), dtype: float64


The mean has dropped substantially and now sits close to the median, confirming the outliers — not a genuinely high-mileage fleet — were what skewed the original average. This is exactly the case B.2 flagged as median-territory: a few extreme values rather than a smooth spread, so the median was the right fill, not the mean.

## Part E — Correcting Data Types

### E.1 — Check Current Data Types

In [15]:
print(df.dtypes)

Car_ID               str
Make                 str
Model                str
Year               int64
Price ($)        float64
Mileage (km)     float64
Fuel Type            str
Transmission         str
Owner History        str
Sold_Yes_No          str
dtype: object


**Exercise E.1 answer:**

Already numeric: `Year`, `Price ($)`, `Mileage (km)` (`int64`/`float64`).
Still `object` (text) and needing encoding before use in a model: `Make`, `Model`, `Fuel Type`, `Transmission`, `Owner History`, `Sold_Yes_No`, plus `Car_ID` (an identifier, not something to encode for modelling).

In [16]:
# E.2 — Enforce numeric types defensively, even though they already look numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Price ($)"] = pd.to_numeric(df["Price ($)"], errors="coerce")
df["Mileage (km)"] = pd.to_numeric(df["Mileage (km)"], errors="coerce")
print(df[["Year", "Price ($)", "Mileage (km)"]].dtypes)

Year              int64
Price ($)       float64
Mileage (km)    float64
dtype: object


In [17]:
# E.3 — Encode categorical columns

# Binary: Sold_Yes_No -> 0/1
df["Sold_Yes_No"] = df["Sold_Yes_No"].map({"Yes": 1, "No": 0})

# Label-encode Fuel Type and Transmission (check the category order before trusting the codes)
df["Fuel Type"] = df["Fuel Type"].astype("category")
print("Fuel Type categories, in code order:", list(df["Fuel Type"].cat.categories))
df["Fuel Type"] = df["Fuel Type"].cat.codes

df["Transmission"] = df["Transmission"].astype("category")
print("Transmission categories, in code order:", list(df["Transmission"].cat.categories))
df["Transmission"] = df["Transmission"].cat.codes

Fuel Type categories, in code order: ['Diesel', 'Electric', 'Hybrid', 'Petrol']
Transmission categories, in code order: ['Automatic', 'Manual']


In [18]:
# One-hot encode Owner History (no natural order between First/Second/Third Owner)
df = pd.get_dummies(df, columns=["Owner History"], drop_first=True)
df.filter(like="Owner History").head()

,Owner History_Second Owner,Owner History_Third Owner
0,True,False
1,False,False
2,True,False
3,False,True
4,False,False


In [19]:
# E.4 — Verify
print(df.dtypes)

Car_ID                            str
Make                              str
Model                             str
Year                            int64
Price ($)                     float64
Mileage (km)                  float64
Fuel Type                        int8
Transmission                     int8
Sold_Yes_No                     int64
Owner History_Second Owner       bool
Owner History_Third Owner        bool
dtype: object


**Exercise E.4 answer:**

- `Year`, `Price ($)` and `Mileage (km)` are `int64`/`float64`, confirmed above.
- `Fuel Type` and `Transmission` are now `int8` (encoded).
- `Owner History_Second Owner` and `Owner History_Third Owner` columns now exist (First Owner is the dropped baseline category — represented implicitly when both new columns are 0).

## Part F — Saving the Cleaned Dataset

In [20]:
# F.1 — Confirm Google Drive is mounted (Colab only; no-op locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    out_path = "/content/drive/My Drive/data_science/Cleaned_Car_Sales_Dataset.csv"
except ImportError:
    out_path = "data/Cleaned_Car_Sales_Dataset.csv"

# F.3 — Save the cleaned DataFrame
df.to_csv(out_path, index=False)
print(f"Cleaned dataset saved successfully to {out_path}")

Cleaned dataset saved successfully to data/Cleaned_Car_Sales_Dataset.csv


In [21]:
# F.4 — Verify the file was saved
import os
print(os.listdir(os.path.dirname(out_path)))

['Car_Sales_Dataset.csv', 'Cleaned_Car_Sales_Dataset.csv']


## Wrap-Up & Submission Checklist

- [x] Dataset imported successfully from Google Drive
- [x] Missing values in `Price ($)` identified and handled, with method justified (Exercise B.2 — mean, since the distribution is roughly symmetric)
- [x] Duplicate rows checked, removed, and verified (Part C)
- [x] Answers to the `Car_ID` duplicate investigation written in a text cell (Exercise C.2)
- [x] Mileage outliers identified and handled (Part D)
- [x] Data types corrected and categorical columns encoded (Part E)
- [x] Cleaned dataset saved back to Google Drive as `Cleaned_Car_Sales_Dataset.csv`